# Study report: kconfig-factors

The study changes one Kconfig factor of the WP3 ranking at a time (a
variant) and measures the three latencies: the IRQ entry (`irq`), the
ISR to thread wake (`isr2thr`) and the thread to thread latency (`t2t`).
Each figure compares a variant with the point of the variant `baseline`
that has the same load, in the same run. The study has no hypotheses of
its own: the four sub-studies (scheduler, tick-mode, code-generation,
stack-guard) have them.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-kconfig-factors@nucleo_g474re"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Variants

Each variant changes one Kconfig factor against the defaults of the
apps: `SIZE_OPTIMIZATIONS`, `SCHED_SIMPLE`, the tickless kernel at 10000
ticks/s, time slicing with 20 ms slices, `MPU_STACK_GUARD=y` and
`ASSERT=n`. The point `baseline` has these defaults. `wp3-variants.md`
describes the variants and two Kconfig traps. The points with a load
are `t64` (64 load timers) and `busy16` (16 busy threads, priority 5).
The table gives the Kconfig of each point, from the design of the
experiments.

In [ ]:
import yaml


def design_points(exp):
    """The points of an experiment and their Kconfig, from its experiment.yaml."""
    f = data.trees.REPO_DIR / inst.experiment_json(exp)["design"]["file"]
    y = yaml.safe_load(f.read_text())
    kv = lambda k: ", ".join(f"{a}={'y' if b is True else 'n' if b is False else b}" for a, b in (k or {}).items())
    rows = [{"point": v, "load": "none", "kconfig": kv((x or {}).get("kconfig")) or "(defaults of the app)"}
            for v, x in y["axes"]["variant"].items()]
    rows += [{"point": p["id"], "load": p["axes"].get("load"), "kconfig": kv(p.get("kconfig"))}
             for p in y.get("points", [])]
    return pd.DataFrame(rows).set_index("point")


LOAD_NAMES = {"none": "no load", "busy16": "16 busy threads", "t64": "64 load timers"}


def diffs(exp):
    """Each point minus the point of the variant baseline with the same load, in the same run (nbfig.diffs)."""
    return nbfig.diffs(inst, exp, "variant", "baseline", match=("load",), names=LOAD_NAMES)

In [ ]:
EXPERIMENTS = ["irq-kconfig-1m", "wake-kconfig-1m", "thread-kconfig-1m"]
shown = [e for e in EXPERIMENTS if nbfig.built(inst, e)]
pd.concat({e: design_points(e) for e in shown}, names=["experiment"])

## How to read the difference figures

Each bar is the value of a point minus the value of the point with the
variant `baseline` and the same load, in the same run. A bar to the left
of 0 means that the variant is faster. The three panels give the p50,
the p99 and the max. Each panel has its own x scale. The colour is the
load of the point. The menu above the figure selects the metric. The
table below each figure gives the same differences in cycles.

## IRQ entry (irq-kconfig-1m)

The edge to the ISR on the wrapped ISR path (`_isr_wrapper`), with an
idle CPU. The IRQ entry path does no scheduling and saves no FP context.
Thus the scheduler variants and `fpu-sharing` are references for "no
change". The points `*-t64` have 64 load timers.

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    d = diffs("irq-kconfig-1m")
    nbfig.diff_fig(d, "irq-kconfig-1m: each variant minus the baseline of the same load").show()

In [ ]:
nbfig.diff_table(d) if nbfig.built(inst, "irq-kconfig-1m") else None

The measured values of each point (not the differences):

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    df = inst.rows("irq-kconfig-1m")
    nbfig.dots(df, "irq-kconfig-1m: the values of each point").show()

In [ ]:
nbfig.pivot(df, ["p50", "p99", "max"]) if nbfig.built(inst, "irq-kconfig-1m") else None

## ISR to thread (wake-kconfig-1m)

The wrapped ISR gives a semaphore, and the woken thread runs. `isr2thr`
is the ISR time stamp to the time stamp of the woken thread, `irq2thr`
is the edge to the woken thread. The points `*-busy16` have 16 busy
threads with priority 5, lower than the priority of the woken thread.
The points `*-t64` have 64 load timers.

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    d = diffs("wake-kconfig-1m")
    nbfig.diff_fig(d, "wake-kconfig-1m: each variant minus the baseline of the same load").show()

In [ ]:
nbfig.diff_table(d) if nbfig.built(inst, "wake-kconfig-1m") else None

The measured values of each point (not the differences):

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    df = inst.rows("wake-kconfig-1m")
    nbfig.dots(df, "wake-kconfig-1m: the values of each point").show()

In [ ]:
nbfig.pivot(df, ["p50", "p99", "max"]) if nbfig.built(inst, "wake-kconfig-1m") else None

## Thread to thread (thread-kconfig-1m)

A thread gives a semaphore, and the taker thread preempts the giver
thread. `t2t` is the wake call to the first instruction of the taker.
`t2t_ret` is the wake call to its return. Each sample has one thread
switch. The experiment has no points with a load.

In [ ]:
if nbfig.built(inst, "thread-kconfig-1m"):
    d = diffs("thread-kconfig-1m")
    nbfig.diff_fig(d, "thread-kconfig-1m: each variant minus the baseline of the same load").show()

In [ ]:
nbfig.diff_table(d) if nbfig.built(inst, "thread-kconfig-1m") else None

The measured values of each point (not the differences):

In [ ]:
if nbfig.built(inst, "thread-kconfig-1m"):
    df = inst.rows("thread-kconfig-1m")
    nbfig.dots(df, "thread-kconfig-1m: the values of each point").show()

In [ ]:
nbfig.pivot(df, ["p50", "p99", "max"]) if nbfig.built(inst, "thread-kconfig-1m") else None